# Package 설치

In [ ]:
# uv venv .venv --python=3.12
# .venv\Scripts\activate

In [ ]:
# !uv pip install ipykernel ipywidgets
# !uv pip install torch
# !uv pip install transformers tokenizers datasets accelerate sentencepiece pillow  timm

# HuggingFace transformers의 Pipeline을 이용한 모델 활용

- Pipeline은 Transformers 라이브러리의 가장 기본적인 객체로, **전처리 - 추론 -> 후처리** 로 이어지는 일련의 과정을 자동화하여 손쉽게 모델을 사용할 수 있게 해준다.
- Task에 따라 다양한 Pipeline 클래스를 제공하며 `pipeline` 함수를 이용해 쉽게 생성할 수 있다.
- **task만 지정**해서 그 task에 대해 기본적으로 제공 모델과 토크나이저를 사용하거나 또는 **직접 사용할 모델과 토크나이저를 지정**해 생성할 수 있다.
  - **토크나이저의 경우 같은 사용할 모델의 ID로 Load하여 그 모델이 학습할 때 사용한 것을 로드한다.**
- https://huggingface.co/docs/transformers/pipeline_tutorial

![huggingface_pipeline.png](figures/huggingface_pipeline.png)

## 지원하는 주요 태스크
- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
### 자연어 처리 태스크
- **text-classification**: 텍스트 분류
- **text-generation**: 텍스트 생성
- **translation**: 번역
- <del>summarization</del>: 요약
- <del>question-answering</del>: 질의응답
- **fill-mask**: 마스크 토큰 채우기
- **token-classification**: 개체명 인식, Pos tagging 같이 개별 토큰에 대한 분류
- **feature-extraction**: 특징 추출(context vector)

### 영상 처리 태스크
- **image-classification**: 이미지 분류
- **object-detection**
  -  객체 검출 (Object Detection)
  -  이미지 안에서 객체들의 위치와 class를 찾아내는 작업
- **image-segmentation**
  -  이미지 세분화 (Image Segmentation)
  -  이미지를 픽셀 단위로 분할하여 각 픽셀이 어떤 객체에 속하는지 분류하는 작업

## 모델 검색
![huggingface_model_search.png](figures/huggingface_model_search.png)



## pipeline 함수
- 주요파라미터
  - **task:** 수행하려는 작업의 유형을 문자열로 지정한다.
  - **model:**
    - 사용할 사전 학습된 모델의 이름 또는 경로를 지정한다.
    - 모델이름(ID)은 `[모델소유자이름]/[모델이름]` 형식이다. Hugging Face에서 제공하는 모델의 경우는 `모델소유자이름`이 생략되어 있다. (ex: "google/gemma-2-2b", "gpt2")
    - 모델을 명시적으로 지정하지 않으면, **task에 맞는 기본 모델이 로드**된다.
  - **tokenizer:** 자연어 task에서 사용할 토크나이저를 지정한다. 생략하면 모델과 같이 제공되는(model과 이름이 같은 토크나이저) 토크나이저를 사용한다.
  - **framework:** 사용할 딥러닝 프레임워크를 지정한다. 'pt'는 PyTorch(Default), 'tf'는 TensorFlow를 지정한다.
  - **device:** Pipeline 모델을 실행할 디바이스를 지정한다. 문자열로 `"cpu", "cuda:1", "mps"`, 또는 GPU 번호를 정수로 지정한다.
  - **revision:** 모델의 특정 버전을 지정할 때 사용한다.
  - **trust_remote_code:** hub 모델을 직접 다운 받는 것이 아니라 모델을 다운 받는 **코드**를 다운 받아 local에서 실행하는 경우 코드를 실행할 수있게 할 지 여부. (bool)
  - **use_fast:**
    - 빠른 토크나이저를 사용할지 여부를 지정합니다. 기본값은 True입니다.
    - 빠른 토크나이저는 `Rust` 언어로 구현되어 속도가 빠르다. 단 모든 모델에 대해 지원하지 않는다. 지원하지 않을 경우 `use_fast=True`로 설정해도 일반 토크나이저가 사용된다.

## Task 별 pipeline 실습

### 텍스트 분류

In [ ]:
from transformers import pipeline

In [ ]:
# task는 필수. model, tokenizer를 생략하면 task의 기본 모델을 사용.
# 처음 사용시 모델/토크나이저 다운로드: C:\Users\<사용자계정>\.cache\huggingface\hub
pipe = pipeline(task="text-classification")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
result = pipe("I am happy.")
result = pipe("I am unhappy.")

In [ ]:
result

[{'label': 'NEGATIVE', 'score': 0.9997568726539612}]

In [ ]:
data = [
    "The project was completed successfully.",
    "She always brings positive energy to the team.",
    "I am confident that we will achieve our goals.",
    "The results were not as expected.",
    "He struggled to meet the deadline.",
    "The client was dissatisfied with the final product."
]

result_list = pipe(data)

In [ ]:
result_list

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [ ]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."
    , "이건 별로에요"
    , "사지 마세요."
]

In [ ]:
result_list2 = pipe(kor_texts)
result_list2


[{'label': 'POSITIVE', 'score': 0.9855567812919617},
 {'label': 'POSITIVE', 'score': 0.7425775527954102},
 {'label': 'POSITIVE', 'score': 0.6555718183517456},
 {'label': 'NEGATIVE', 'score': 0.5247920155525208},
 {'label': 'POSITIVE', 'score': 0.9307546615600586},
 {'label': 'POSITIVE', 'score': 0.7802650332450867}]

In [ ]:
model = "tabularisai/multilingual-sentiment-analysis"
pipe = pipeline(task='text-classification', model=model, tokenizer=model)
# tokenizer 설정 생략 -> model의 id를 같이 사용.

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
kor_texts

['이 영화 정말 재미있어요!',
 '서비스가 별로였어요.',
 '제품 품질이 우수합니다.',
 '따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요.',
 '이건 별로에요',
 '사지 마세요.']

In [ ]:
result_list = pipe(kor_texts)
result_list

[{'label': 'Positive', 'score': 0.7790912389755249},
 {'label': 'Negative', 'score': 0.8723201155662537},
 {'label': 'Positive', 'score': 0.6111961007118225},
 {'label': 'Positive', 'score': 0.44615432620048523},
 {'label': 'Negative', 'score': 0.7431011199951172},
 {'label': 'Positive', 'score': 0.6710690855979919}]

### 제로샷 분류
- 제로샷(Zero-shot)은 각 개별 작업에 대한 특정 교육 없이 작업을 수행할 수 있는 task다.
- 입력 텍스트와 함께 클래스 레이블을 제공하면 분류 작업을 한다.
- 모델은  `task`에서 `Zero-Shot` 으로 시작하는 task를 선택하여 검색한다.

In [ ]:
text = ["Python is a programming language.",
        "I love soccer",
        "The stock price rose slightly today."]

labels1 = ["IT", "Sports"]
labels2 = ["business", "programming", "sports", "movie", "education"]

In [ ]:
model = "facebook/bart-large-mnli"
pipe = pipeline("zero-shot-classification", model=model)
result = pipe(text, candidate_labels=labels1)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
result

[{'sequence': 'Python is a programming language.',
  'labels': ['IT', 'Sports'],
  'scores': [0.575852632522583, 0.424147367477417]},
 {'sequence': 'I love soccer',
  'labels': ['Sports', 'IT'],
  'scores': [0.9935312867164612, 0.006468693260103464]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['IT', 'Sports'],
  'scores': [0.6849519610404968, 0.3150480091571808]}]

In [ ]:
result = pipe(text, candidate_labels=labels2)

In [ ]:
result

[{'sequence': 'Python is a programming language.',
  'labels': ['programming', 'business', 'movie', 'sports', 'education'],
  'scores': [0.9856367111206055,
   0.005072700325399637,
   0.003402342554181814,
   0.002961937105283141,
   0.002926240209490061]},
 {'sequence': 'I love soccer',
  'labels': ['sports', 'programming', 'business', 'movie', 'education'],
  'scores': [0.9952406287193298,
   0.0012840843992307782,
   0.0012676483020186424,
   0.0012649564305320382,
   0.0009427035693079233]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['business', 'movie', 'programming', 'sports', 'education'],
  'scores': [0.7462776303291321,
   0.06974834948778152,
   0.06889285147190094,
   0.06450819969177246,
   0.05057302862405777]}]

### 텍스트 생성

In [1]:
from transformers import pipeline
pipe = pipeline(task="text-generation")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

In [2]:
start_text = "Today weather"
txt = pipe(start_text)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [4]:
txt[0]['generated_text']

'Today weather forecasters will be watching the weather carefully because today is the start of the hurricane season. This season will begin June 1 and run through November 30. Hurricanes are a natural occurrence, but they can be devastating. They can cause damage to homes, businesses, and transportation systems. They can also kill people and animals. The National Hurricane Center, part of the National Oceanic and Atmospheric Administration, is responsible for monitoring and predicting hurricanes. They issue advisories and warnings to help people prepare for and respond to hurricanes.\n\nHurricanes are large, rotating storms that form over warm ocean waters. They get their energy from the heat and moisture in the air. As a hurricane moves over land, it can bring heavy rain, strong winds, and storm surges. Storm surges are large waves that can cause flooding. Hurricanes can also bring tornadoes and flash floods.\n\nThe United States is prone to hurricanes because of its location near th

In [6]:
result = pipe(["I am a", "Python is", "LLM is"], max_new_tokens=500)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [18]:
print(result[2][0]['generated_text'])

LLM is an abbreviation for Law Masters. It is a postgraduate law degree that is studied after a bachelor's degree. The law degree is required in many areas of the law, including litigation, corporate law, intellectual property law, and tax law.

An LLM is a specialized degree that focuses on a specific area of law. It is often pursued by students who wish to specialize in a particular area of law, such as international law, commercial law, or criminal law.

To pursue an LLM, students typically need to have a bachelor's degree in law or a related field and a minimum of 3 years of legal experience. The LLM program usually takes one year to complete and may be completed either full-time or part-time.

An LLM can be very beneficial for those who want to specialize in a particular area of law and advance their careers. It can also help students to gain the knowledge and skills needed to take on more complex and senior roles in the legal field.

Some of the benefits of pursuing an LLM includ

In [19]:
result = pipe("나는 어제 집에 가다가")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [20]:
print(result[0]['generated_text'])

나는 어제 집에 가다가 보니까 거기에 사는 아이가 엄마 없이 혼자 있더라. 그 아이가 5살 정도 되던가. 그 아이는 엄마가 어디로 갔는지 모르고, 엄마가 없다고 생각했는지 모르겠다. 어린 아이는 엄마 없이 집에 혼자 있으면서 무엇을 하는지 몰라서 조금 신경이 쓰여서 그 집으로 갔었다. 그 집에서 사는 아이가 5살 정도 되던가. 그 아이는 엄마가 어디로 갔는지 모르고, 엄마가 없다고 생각했는지 모르겠다. 어린 아이는 엄마 없이 집에 혼자 있으면서 무엇을 하는지 몰라서 조금 신경이 쓰여서 그 집으로 갔었다. 그 집에서 사는 아이가 5살 정도 되던가. 그 아이는 엄마가 어디로 갔는지 모르고, 엄마가 없다고 생각했는지 모르겠다. 어린 아이는 엄마 없이 집에 혼자 있으면서 무엇을 하는지 몰라서 조금 신경이 쓰여서 그 집으로 갔었다. 그 집에서 사는 아이가 5살 정도 되던가. 그 아이는 엄마가 어디로 갔는지 모르고,


### 마스크 채우기

In [21]:
text = "I'm going to <mask> because <mask> am hurt."

model = "FacebookAI/xlm-roberta-base"
pipe = pipeline(task='fill-mask', model=model)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [23]:
result = pipe(text, top_k=2)
# <mask>에 들어갈 확률이 가장 높은 단어 top_k개를 반환(기본: 5)
result

[[{'score': 0.41619235277175903,
   'token': 91190,
   'token_str': 'cry',
   'sequence': "<s> I'm going to cry because<mask> am hurt.</s>"},
  {'score': 0.17670488357543945,
   'token': 68,
   'token_str': 'die',
   'sequence': "<s> I'm going to die because<mask> am hurt.</s>"}],
 [{'score': 0.9929056167602539,
   'token': 87,
   'token_str': 'I',
   'sequence': "<s> I'm going to<mask> because I am hurt.</s>"},
  {'score': 0.0060635339468717575,
   'token': 17,
   'token_str': 'i',
   'sequence': "<s> I'm going to<mask> because i am hurt.</s>"}]]

In [24]:
text = "오늘 밤은 전국이 흐린 가운데 대부분 지역에 <mask>가 내리겠고, 기온이 내려가면서 점차 <mask>이 오는 곳이 많겠습니다"
result = pipe(text, top_k=2)
result

[[{'score': 0.8498658537864685,
   'token': 7091,
   'token_str': '비',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.10534586757421494,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 눈 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'}],
 [{'score': 0.6368799805641174,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 눈 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.1474808305501938,
   'token': 208400,
   'token_str': '구름',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 구름 이 오는 곳이 많겠습니다</s>'}]]

### Token별 분류
- task: token-classification
  - 개체명인식(ner), 품사부착(pos tagging)을 수행하는 task
  - 개체명 인식은 문장에서 특정한 개체명(예: 사람 이름, 지명, 조직명 등)을 식별하는 task이다.

In [25]:
text = "My name is Sylvain and I work at Hugging Face in Brooklyn."

In [26]:
model = "dbmdz/bert-large-cased-finetuned-conll03-english"

pipe = pipeline(task="token-classification", model=model)

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

In [27]:
pipe(text)

[{'entity': 'I-PER',
  'score': np.float32(0.99938285),
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': np.float32(0.99815494),
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': np.float32(0.9959072),
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.99923277),
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': np.float32(0.9738931),
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': np.float32(0.976115),
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': np.float32(0.9887976),
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': np.float32(0.9932106),
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

### 질의 응답
- 문서와 질문을 주면 문서에서 답을 찾아 응답한다.

In [28]:
model = "Qwen/Qwen2.5-0.5B-Instruct"
pipe = pipeline(task="text-generation")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [34]:
question="Where do I work?"
# question="Where is Hugging Face?"

context="My name is Sylvain and I work at Hugging Face in Brooklyn"
# instruction 모델 -> 대화(User와 Model(AI))
# 대화메세지 : role - 대화를 하는 발화자(user, assistant)
message = [
    {
        "role":"user",
        "content":f"""
다음에 나오는 <문서>의 내용을 바탕으로 <질문>에 대해서 답변해줘.
<문서>에 질문에 대해 참고할 내용이 없으면 "모른다" 라고 답해줘.

<문서>
{context}
</문서>

<질문>
{question}
</질문>
"""
    }
]

result = pipe(message, max_new_tokens=300)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [37]:
print(result[0]["generated_text"][0]["content"])


다음에 나오는 <문서>의 내용을 바탕으로 <질문>에 대해서 답변해줘. 
<문서>에 질문에 대해 참고할 내용이 없으면 "모른다" 라고 답해줘.

<문서>
My name is Sylvain and I work at Hugging Face in Brooklyn
</문서>

<질문>
Where do I work?
</질문>



In [38]:
print(result[0]["generated_text"][1]["content"])

<think>
Okay, let's see. The user is asking where I work based on the provided document. The document says, "My name is Sylvain and I work at Hugging Face in Brooklyn." So, the person's name is Sylvain, and they work at Hugging Face in Brooklyn. 

The question is straightforward: "Where do I work?" The document clearly states the workplace, which is Hugging Face in Brooklyn. I need to make sure there's no ambiguity here. The document is written in a way that directly answers the question. There's no other information given about where Sylvain works besides Hugging Face in Brooklyn. 

I should check if there's any trick or if the user is trying to get me to say something else, but based on the information provided, the answer is straightforward. The user might be testing if I can extract the correct information from the document. Since the document is in the same language as the question (English), there's no translation needed. 

So, the correct answer should be "Hugging Face in Brookl

In [39]:
context = """우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.
"""

question1 = "현대차 기아의 미국 수출비중은?"
question2 = "자동차 수출이 얼마나 급감했나?"
question3 = "대미 수출 감소에 국내 자동차 업체들의 대응방법은?"

In [50]:
message = [
    {
        "role":"user",
        "content":f"""
다음에 나오는 <문서>의 내용을 바탕으로 <질문>에 대해서 답변해줘.
<문서>에 질문에 대해 참고할 내용이 없으면 "모른다" 라고 답해줘.

<문서>
{context}
</문서>

<질문>
짜장면 조리법을 알려줘.
</질문>
"""
    }
]
print(message[0]['content'])


다음에 나오는 <문서>의 내용을 바탕으로 <질문>에 대해서 답변해줘. 
<문서>에 질문에 대해 참고할 내용이 없으면 "모른다" 라고 답해줘.

<문서>
우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.

</문서>

<질문>
짜장면 조리법을 알려줘.
</질문>



In [51]:
result = pipe(message, max_new_tokens=1000)

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [52]:
result[0]["generated_text"][0]

{'role': 'user',
 'content': '\n다음에 나오는 <문서>의 내용을 바탕으로 <질문>에 대해서 답변해줘. \n<문서>에 질문에 대해 참고할 내용이 없으면 "모른다" 라고 답해줘.\n\n<문서>\n우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.\n1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.\n4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.\n국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.\n\n</문서>\n\n<질문>\n짜장면 조리법을 알려줘.\n</질문>\n'}

In [53]:
print(result[0]["generated_text"][1]['content'])

<think>
Okay, let's see. The user provided a document about Korea's top two exports, cars, and the impact of US tariffs. Then they asked for the recipe for jjajangmyeon. Hmm, the document is about car exports and tariffs, which is completely unrelated to the question about jjajangmyeon. The user might have made a mistake in the document or the question. Since the document doesn't mention anything about cooking methods or recipes, and the question is about jjajangmyeon, I need to check if there's any connection.

First, I'll look through the document again. It talks about the automotive industry in Korea, tariffs affecting exports, and the companies' strategies. There's no mention of food, cooking, or recipes. The question is about jjajangmyeon, which is a Korean dish made with black bean sauce, rice, and meat. The document provided doesn't have any information related to this topic. 

Since there's no relevant information in the document to answer the question, I should respond by sayi

##### 문서 요약

In [62]:
message = [
    {
        "role":"user",
        "content": f"""다음 문서의 내용을 50글자 이내로 한국어로 요약해줘.
<문서>
{context}
</문서>
"""
    }
]
print(message[0]['content'])

다음 문서의 내용을 50글자 이내로 한국어로 요약해줘.
<문서>
우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.

</문서>



In [63]:
res = pipe(message, max_new_tokens=1000)

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [64]:
print(res[0]["generated_text"][1]['content'])

<think>
Okay, I need to summarize the given article in Korean, keeping it under 50 characters. Let me read through the article first to understand the key points.

The article mentions that South Korea's top two export products, cars, are experiencing significant drops in exports due to U.S. tariffs under Trump's administration. The government data shows a 4.4% decrease in May, with exports to the U.S. dropping by 32%. This follows a 25% tariff on imported cars in April and a 25% tariff on auto parts starting in May. The industry is worried that the impact will worsen if tariffs remain long-term. Companies like Hyundai and Kia are considering price increases and expanding local production to cope with the situation, which could further reduce their U.S. sales and exports.

Now, I need to condense this into a short summary. The main points are:

1. South Korea's top two exports (cars) are suffering from U.S. tariffs.
2. May's exports decreased by 4.4%, with a 32% drop in U.S. sales.
3. 

##### 번역

In [68]:
language = "중국어"
message = [
    {
        "role":"user",
        "content": f"""다음 내용을 {language} 로 번역해줘.
<번역할내용>
{context}
</번역할내용>
"""
    }
]
print(message)

[{'role': 'user', 'content': '다음 내용을 중국어 로 번역해줘.\n<번역할내용>\n우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.\n1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.\n4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.\n국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.\n\n</번역할내용>\n'}]


In [72]:
result = pipe(message, max_new_tokens=3000)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=3000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [67]:
print(result[0]["generated_text"][1]['content'])

<think>
Okay, I need to translate the given Korean text into English. Let me start by reading through the Korean content carefully to understand the context and key points. The article is discussing the impact of US tariffs on South Korea's top two export items, specifically automobiles, under President Trump's administration. The main concern is the significant drop in exports to the US, which accounts for the largest share of South Korea's car exports. The government's statistics show a 4.4% decrease in car exports to USD 62 billion, with a 32.0% drop specifically to the US. This follows the 25% tariffs on imported cars in April and the extension to some automotive parts starting in May. The article mentions that if the tariffs remain long-term, the damage will escalate uncontrollably. It also notes that both Hyundai and Kia, as the top two car manufacturers, are increasing their domestic production while considering price hikes to absorb the tariff impact. If price increases occur, 

In [73]:
print(result[0]["generated_text"][1]['content'])

<think>
Okay, I need to translate the given English text into Chinese. Let me start by reading the English content carefully to understand the context and key points.

The main topic is about South Korea's two largest export items, cars, experiencing a significant decline in exports due to tariffs imposed by the Trump administration. The concern is that if the tariffs remain long-term, the damage will be uncontrollable. The article mentions that the two top automakers, Hyundai and Kia, have a market share of up to 85% in the U.S., which could lead to more severe consequences if tariffs persist.

Looking at the data provided: in May, total South Korean exports dropped by 4.4% to $62 billion, with exports to the U.S. plummeting by 32% to $18.4 billion. This is after the 25% import duty on cars in April and the extension to some auto parts in May. The article suggests that if the tariffs last longer, the impact will worsen.

Hyundai and Kia are considering increasing production in the U.S